# CITCA 2026 — Fashion-MNIST: comparación de clasificadores base
**MLP, CNN, SVM, KNN y Random Forest | Dataset completo**

Objetivo: establecer referencias experimentales sin búsqueda de hiperparámetros, NAS ni compresión. Activar GPU en Colab para las redes neuronales. Los clasificadores sklearn se ejecutan en CPU. SVM y KNN sobre 60 000 muestras pueden tardar considerablemente. Ejecutar en orden. No se usa el test para elegir configuraciones. Las configuraciones son fijas, no se afirma que sean óptimas.

In [ ]:
import time, shutil, json, platform, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from IPython.display import display
SEED=42
EPOCHS=4
BATCH=64
keras.utils.set_random_seed(SEED)
OUT=Path('fashion_baselines'); OUT.mkdir(exist_ok=True)
print('CPU:',platform.processor(),'GPU:',tf.config.list_physical_devices('GPU'))
print('TensorFlow:',tf.__version__)

## 1. Dataset completo
60 000 imágenes para ajuste y 10 000 para evaluación independiente. No se reserva validación porque no hay selección de modelos ni parada temprana.

In [ ]:
(X,Y),(T,U)=keras.datasets.fashion_mnist.load_data()
X=X.astype('float32')/255; T=T.astype('float32')/255
classes=['Camiseta','Pantalón','Suéter','Vestido','Abrigo','Sandalia','Camisa','Tenis','Bolso','Botín']
assert len(X)==60000 and len(T)==10000
print(X.shape,T.shape,np.bincount(Y),np.bincount(U))
fig,axs=plt.subplots(2,5,figsize=(12,5))
for i,ax in enumerate(axs.flat):
    ax.imshow(X[i],cmap='gray'); ax.set_title(classes[Y[i]]); ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Funciones de evaluación
Exactitud, F1 macro, tiempo de ajuste, tiempo de predicción por lote, latencia individual p50/p95 y tamaño en disco. La latencia individual incluye HOG cuando corresponde y transferencia de salida a CPU en Keras. Se usan 100 imágenes para latencia y las 10 000 para exactitud. El tamaño de KNN incluye sus datos almacenados; Keras incluye estado del optimizador. No es una comparación de hardware homogéneo.

In [ ]:
results=[]; predictions={}; histories={}; models={}
idx=np.random.default_rng(SEED).choice(len(T),100,replace=False)
def evaluate(name,model,pred,fit_s,path,one,prep_train=0.,prep_test=0.,batch_s=0.,params=None):
    predictions[name]=pred
    for i in idx[:5]: one(T[i])
    times=[]
    for i in idx:
        t=time.perf_counter(); one(T[i]); times.append((time.perf_counter()-t)*1000)
    results.append(dict(modelo=name,accuracy=accuracy_score(U,pred),f1_macro=f1_score(U,pred,average='macro'),fit_s=fit_s,prep_train_s=prep_train,prep_test_s=prep_test,predict_test_s=batch_s,latencia_p50_ms=np.median(times),latencia_p95_ms=np.percentile(times,95),parametros=params,tamano_MiB=path.stat().st_size/2**20))
    print(results[-1])

## 3. MLP original
784 → 64 ReLU → 32 ReLU → 10 softmax. Sin dropout. Adam, learning rate 0.001, cuatro épocas. Batch fijo de 64 para ambas redes.

In [ ]:
mlp=keras.Sequential([layers.Input((28,28)),layers.Flatten(),layers.Dense(64,activation='relu'),layers.Dense(32,activation='relu'),layers.Dense(10,activation='softmax')],name='MLP')
mlp.summary()

In [ ]:
def train_nn(name,model,channel=False):
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    a=X[...,None] if channel else X; z=T[...,None] if channel else T
    t=time.perf_counter(); h=model.fit(a,Y,epochs=EPOCHS,batch_size=BATCH,verbose=1); fit=time.perf_counter()-t
    histories[name]=h.history
    t=time.perf_counter(); pred=model.predict(z,batch_size=256,verbose=0).argmax(axis=1); batch_s=time.perf_counter()-t
    path=OUT/(name+'.keras'); model.save(path); models[name]=model
    def one(im):
        v=im[None,...,None] if channel else im[None,...]
        return model(v,training=False).numpy().argmax(axis=1)
    evaluate(name,model,pred,fit,path,one,batch_s=batch_s,params=model.count_params())
train_nn('MLP',mlp)

## 4. CNN base
Dos bloques convolucionales con padding same, pooling, capa densa y dropout fijo de 0.2, como en el cuaderno anterior. No se prueban otras configuraciones.

In [ ]:
keras.utils.set_random_seed(SEED)
cnn=keras.Sequential([layers.Input((28,28,1)),layers.Conv2D(16,3,padding='same',activation='relu'),layers.MaxPooling2D(2),layers.Conv2D(32,3,padding='same',activation='relu'),layers.MaxPooling2D(2),layers.Flatten(),layers.Dense(64,activation='relu'),layers.Dropout(0.2),layers.Dense(10,activation='softmax')],name='CNN')
cnn.summary()
train_nn('CNN',cnn,True)

## 5. Extracción HOG
Se conserva el descriptor del cuaderno anterior. Se extraen características de todas las imágenes, sin ajuste sobre el test. La comparación es entre pipelines completos, no únicamente clasificadores.

In [ ]:
HOG_CFG=dict(orientations=9,pixels_per_cell=(4,4),cells_per_block=(2,2),block_norm='L2-Hys')
def features(images):
    return np.asarray([hog(im,**HOG_CFG) for im in images],dtype=np.float32)
t=time.perf_counter(); F=features(X); prep_train=time.perf_counter()-t
t=time.perf_counter(); G=features(T); prep_test=time.perf_counter()-t
print(F.shape,G.shape,prep_train,prep_test)
(OUT/'hog_config.json').write_text(json.dumps(HOG_CFG,indent=2))

## 6. Clasificadores tradicionales
Cada modelo se ajusta una sola vez. HOG se comparte, pero su costo se reporta para cada pipeline. SVM RBF puede tardar varios minutos o más; KNN almacena las muestras y realiza la mayor parte del trabajo al predecir.

In [ ]:
def train_classic(name,model):
    t=time.perf_counter(); model.fit(F,Y); fit=time.perf_counter()-t
    t=time.perf_counter(); pred=model.predict(G); batch_s=time.perf_counter()-t
    path=OUT/(name+'.joblib'); joblib.dump(model,path); models[name]=model
    evaluate(name,model,pred,fit,path,lambda im:model.predict(features(im[None,...])),prep_train,prep_test,batch_s)

In [ ]:
train_classic('KNN',KNeighborsClassifier(n_neighbors=5,weights='uniform',metric='euclidean',n_jobs=2))

In [ ]:
train_classic('SVM',SVC(C=10,kernel='rbf',gamma='scale',cache_size=1024))

In [ ]:
train_classic('Random_Forest',RandomForestClassifier(n_estimators=100,max_features='sqrt',random_state=SEED,n_jobs=2))

## 7. Comparación experimental
Los resultados son calculados, no estimados. El costo total de entrenamiento incluye extracción HOG cuando corresponde.

In [ ]:
df=pd.DataFrame(results)
df['entrenamiento_total_s']=df.fit_s+df.prep_train_s
df['test_total_s']=df.predict_test_s+df.prep_test_s
df=df.sort_values('accuracy',ascending=False).reset_index(drop=True)
display(df.round(4))
df.to_csv(OUT/'comparacion.csv',index=False)
for col,label in [('accuracy','Exactitud (%)'),('f1_macro','F1 macro'),('entrenamiento_total_s','Entrenamiento total (s)'),('latencia_p50_ms','Latencia p50 (ms)'),('tamano_MiB','Tamaño (MiB)')]:
    fig,ax=plt.subplots(figsize=(8,4)); vals=df[col]*100 if col=='accuracy' else df[col]
    bars=ax.barh(df.modelo,vals); ax.bar_label(bars,fmt='%.3f',padding=3); ax.margins(x=.2); ax.set_xlabel(label); ax.invert_yaxis(); fig.tight_layout(); fig.savefig(OUT/(col+'.png'),dpi=150); plt.show()

In [ ]:
for name,pred in predictions.items():
    fig,ax=plt.subplots(figsize=(8,7)); ConfusionMatrixDisplay.from_predictions(U,pred,display_labels=classes,xticks_rotation=45,ax=ax,colorbar=False); ax.set_title(name); fig.tight_layout(); fig.savefig(OUT/(name+'_confusion.png'),dpi=150); plt.show()
    report=classification_report(U,pred,target_names=classes,output_dict=True,zero_division=0)
    pd.DataFrame(report).T.to_csv(OUT/(name+'_reporte.csv'))
for metric in ['accuracy','loss']:
    fig,ax=plt.subplots(figsize=(7,4))
    for name,h in histories.items(): ax.plot(range(1,len(h[metric])+1),h[metric],marker='o',label=name)
    ax.set_xlabel('Época'); ax.set_ylabel(metric+' de entrenamiento'); ax.legend(); plt.show()

## 8. Recargar modelos e inferir una imagen
Los modelos Keras reciben píxeles normalizados. Los modelos joblib reciben HOG con la configuración guardada. Cargar únicamente archivos joblib propios o de confianza.

In [ ]:
i=17; im=T[i]; plt.imshow(im,cmap='gray'); plt.title('Real: '+classes[U[i]]); plt.axis('off'); plt.show()
for name in ['MLP','CNN']:
    model=keras.models.load_model(OUT/(name+'.keras'))
    v=im[None,...,None] if name=='CNN' else im[None,...]
    print(name,classes[model(v,training=False).numpy().argmax()])
for name in ['KNN','SVM','Random_Forest']:
    model=joblib.load(OUT/(name+'.joblib')); print(name,classes[model.predict(features(im[None,...]))[0]])

## 9. Exportar resultados
Descargar ZIP con modelos, configuración HOG, tablas CSV y gráficas. Preguntas: ¿qué modelo obtiene mayor exactitud? ¿Cuál tiene menor latencia? ¿Qué clases se confunden? ¿Qué efecto tiene el presupuesto fijo de cuatro épocas? Una sola ejecución no demuestra superioridad universal.

In [ ]:
pd.DataFrame({'real':U,**predictions}).to_csv(OUT/'predicciones_test.csv',index=False)
(OUT/'config.json').write_text(json.dumps(dict(seed=SEED,epochs=EPOCHS,batch=BATCH,train=len(X),test=len(T)),indent=2))
zip_path=shutil.make_archive('FashionMNIST_baselines','zip',OUT)
print(zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('ZIP disponible localmente')